In [1]:
%pip install tensordev

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install --user --no-build-isolation git+https://github.com/crispitagorico/sigkernel.git

  Cloning https://github.com/crispitagorico/sigkernel.git to /tmp/pip-req-build-yskvndbb
  Running command git clone --filter=blob:none --quiet https://github.com/crispitagorico/sigkernel.git /tmp/pip-req-build-yskvndbb
  Resolved https://github.com/crispitagorico/sigkernel.git to commit 40a583155ea8d2194af0e90dddab37e2659cfcfd
  Preparing metadata (pyproject.toml) ... done
  Created wheel for sigkernel: filename=sigkernel-0.0.1-cp312-cp312-linux_x86_64.whl size=112104 sha256=c9dd49c6c3007f60baf6e7d9f828c8a8ba8bfb98df854539db55a903b76fd5f8
  Stored in directory: /tmp/pip-ephem-wheel-cache-45ziirl9/wheels/93/5a/ce/86d6d28e87b16c853dfb46c3ad44c23e4beacd9fcf1b8bc404
Successfully built sigkernel

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


# Classification of UEA time-series datasets

This notebook provides a lightweight interface for testing the Volterra signature kernel on UEA time-series classification tasks.

The full experimental pipeline used to reproduce the results from the paper is implemented in `run_classifier.py` and can be launched from the final cell. Since the full run trains over all selected datasets and performs hyperparameter optimization, it may take a substantial amount of time.

For quick experimentation, the next cells expose a reduced version of the pipeline with simpler settings: fewer datasets, fewer Optuna trials, and a smaller hyperparameter search space. This is intended for debugging, sanity checks, and interactive exploration before running the full experiment.

In [1]:
import os
os.environ["XLA_FLAGS"] = "--xla_force_host_platform_device_count=8"

In [3]:
# If JAX was already imported in this notebook before running this cell,
# restart the kernel first so the env vars below actually take effect.

import os
import gc
import json
import csv
import time
import socket
import faulthandler
from pathlib import Path
from datetime import datetime
from typing import Optional, Dict, Any, List

faulthandler.enable(all_threads=True)


import numpy as np
import joblib
import torch

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.svm import SVC

from tslearn.datasets import UCR_UEA_datasets

import optuna
from optuna.samplers import TPESampler
from tqdm import tqdm

import jax
import jax.numpy as jnp
from jax import config
config.update("jax_enable_x64", True)

import sigkernel as SIG

import run_classifiers as rc

In [4]:
# ============================================================
# Small notebook test config
# ============================================================

DATASETS = [
    "Libras",
]

TRANSFORMS = [(True, False)]
SCALING_METHODS = ["std"]
METHOD_SPECS = [
    {"static_kernel_kind": "linear"},
    # {"static_kernel_kind": "rbf"},
]

N_TRIALS = 30          # small notebook test
N_STARTUP_TRIALS = 2

# Patch the globals used inside run_classifier.py helpers
rc.TRANSFORMS = TRANSFORMS
rc.SCALING_METHODS = SCALING_METHODS
rc.METHOD_SPECS = METHOD_SPECS
rc.OPTUNA_N_TRIALS = N_TRIALS
rc.OPTUNA_STARTUP_TRIALS = N_STARTUP_TRIALS

rc.OPTUNA_STATE_RANK_CHOICES = [1,2] #the parameter R
rc.OPTUNA_DYADIC_CHOICES = [0] #number of dyadic refinments

rc.VSIG_OUTSIDE_WARMUP = True   # faster for notebook testing

In [5]:
# ============================================================
# Minimal notebook test run
# ============================================================

results = []

for name in tqdm(DATASETS, desc="datasets"):
    x_train_raw, y_train_raw, x_test_raw, y_test_raw = (
        rc.UCR_UEA_datasets(use_cache=True).load_dataset(name)
    )

    if any(v is None for v in [x_train_raw, y_train_raw, x_test_raw, y_test_raw]):
        print(f"Skipping {name}: one split is None.")
        continue

    le = LabelEncoder()
    y_train = le.fit_transform(y_train_raw)
    y_test = le.transform(y_test_raw)

    for at, ll in TRANSFORMS:
        x_train_torch_by_scaling = {}
        x_test_torch_by_scaling = {}
        subsample_by_scaling = {}

        # ----------------------------------------------------
        # Prepare scaled/transformed data
        # ----------------------------------------------------
        for scaling_kind in SCALING_METHODS:
            x_train_scaled, x_test_scaled = rc.scale_train_test(
                x_train_raw,
                x_test_raw,
                scaling_kind=scaling_kind,
            )

            x_train = rc.SIG.transform(x_train_scaled, at=at, ll=ll, scale=0.1)
            x_test = rc.SIG.transform(x_test_scaled, at=at, ll=ll, scale=0.1)

            subsample = max(int(np.floor(x_train.shape[1] / 149)), 1)

            x_train = x_train[:, ::subsample, :]
            x_test = x_test[:, ::subsample, :]

            device_vol, dtype_vol = rc.choose_device_and_dtype_volterra(x_train)

            x_train_torch_by_scaling[scaling_kind] = torch.tensor(
                x_train,
                dtype=dtype_vol,
                device=device_vol,
            )
            x_test_torch_by_scaling[scaling_kind] = torch.tensor(
                x_test,
                dtype=dtype_vol,
                device=device_vol,
            )

            subsample_by_scaling[scaling_kind] = subsample

            print(
                f"[prepared] {name} | scaling={scaling_kind} "
                f"train={x_train.shape} test={x_test.shape} subsample={subsample}"
            )

        # ----------------------------------------------------
        # Run each kernel spec
        # ----------------------------------------------------
        for spec in METHOD_SPECS:
            static_kernel_kind = spec["static_kernel_kind"]
            mkey = rc.method_key(static_kernel_kind)

            sampler = TPESampler(
                seed=rc.OPTUNA_SEED,
                n_startup_trials=rc.OPTUNA_STARTUP_TRIALS,
            )

            study = optuna.create_study(direction="maximize", sampler=sampler)

            objective = rc.make_volterra_objective(
                x_train_torch_by_scaling=x_train_torch_by_scaling,
                y_train=y_train,
                static_kernel_kind=static_kernel_kind,
            )

            print(f"\n[optuna] dataset={name}, kernel={mkey}, trials={N_TRIALS}")

            study.optimize(
                objective,
                n_trials=N_TRIALS,
                gc_after_trial=True,
                show_progress_bar=False,
            )

            completed_trials = [
                t for t in study.trials
                if t.state == optuna.trial.TrialState.COMPLETE
            ]

            if len(completed_trials) == 0:
                print(f"[warning] no successful trial for {name} | {mkey}")
                continue

            best_params = dict(study.best_trial.params)

            best_scaling_kind = str(best_params["scaling_kind"])
            best_state_rank = int(best_params["state_rank"])
            train_cv_score = float(study.best_value)

            x_train_torch_best = x_train_torch_by_scaling[best_scaling_kind]
            x_test_torch_best = x_test_torch_by_scaling[best_scaling_kind]

            print("\n[best]")
            print(f"dataset : {name}")
            print(f"kernel  : {mkey}")
            print(f"cv      : {train_cv_score:.4f}")
            print(f"params  : {best_params}")

            # ------------------------------------------------
            # Refit on full training set
            # ------------------------------------------------
            with torch.no_grad():
                G_train, resolved_params = rc.compute_vsig_train_gram(
                    x_train_torch=x_train_torch_best,
                    state_rank=best_state_rank,
                    static_kernel_kind=static_kernel_kind,
                    dyadic_order=int(best_params["dyadic_order"]),
                    sigma=best_params.get("sigma", None),

                    lambda_base=best_params.get("lambda_base", None),
                    alpha_scale=best_params.get("alpha_scale", None),

                    lambda1=best_params.get("lambda1", None),
                    lambda2=best_params.get("lambda2", None),
                    coupling=best_params.get("coupling", None),
                    alpha1=best_params.get("alpha1", None),
                    alpha2=best_params.get("alpha2", None),
                )

            if not rc.gram_is_usable(G_train):
                print(f"[warning] unusable train Gram for {name} | {mkey}")
                continue

            clf = SVC(
                C=float(best_params["C"]),
                kernel="precomputed",
                decision_function_shape="ovo",
            )

            clf.fit(G_train, y_train)

            # ------------------------------------------------
            # Test accuracy
            # ------------------------------------------------
            with torch.no_grad():
                G_test, _ = rc.compute_vsig_test_gram(
                    x_train_torch=x_train_torch_best,
                    x_test_torch=x_test_torch_best,
                    state_rank=best_state_rank,
                    static_kernel_kind=static_kernel_kind,
                    dyadic_order=int(best_params["dyadic_order"]),
                    sigma=best_params.get("sigma", None),

                    lambda_base=best_params.get("lambda_base", None),
                    alpha_scale=best_params.get("alpha_scale", None),

                    lambda1=best_params.get("lambda1", None),
                    lambda2=best_params.get("lambda2", None),
                    coupling=best_params.get("coupling", None),
                    alpha1=best_params.get("alpha1", None),
                    alpha2=best_params.get("alpha2", None),
                )

            if not rc.gram_is_usable(G_test):
                print(f"[warning] unusable test Gram for {name} | {mkey}")
                continue

            test_score = float(clf.score(G_test, y_test))

            print(
                f"[result] dataset={name} kernel={mkey} "
                f"train_cv={train_cv_score:.4f} test={test_score:.4f}"
            )

            results.append(
                {
                    "dataset": name,
                    "kernel": mkey,
                    "scaling": best_scaling_kind,
                    "state_rank": best_state_rank,
                    "train_cv_accuracy": train_cv_score,
                    "test_accuracy": test_score,
                    "subsample": subsample_by_scaling[best_scaling_kind],
                    "best_params": best_params,
                }
            )

            del G_train, G_test, clf, study
            gc.collect()
            rc.maybe_cleanup(force=True)

        del x_train_torch_by_scaling, x_test_torch_by_scaling
        gc.collect()
        rc.maybe_cleanup(force=True)

results_df = pd.DataFrame(results)

print("\nFinal notebook test accuracies:")
display(results_df[[
    "dataset",
    "kernel",
    "scaling",
    "state_rank",
    "train_cv_accuracy",
    "test_accuracy",
    "subsample",
]])

datasets:   0%|                                           | 0/1 [00:00<?, ?it/s]

[prepared] Libras | scaling=std train=(180, 45, 3) test=(180, 45, 3) subsample=1

[optuna] dataset=Libras, kernel=linear, trials=30


[W 2026-05-19 18:33:34,707] Trial 0 failed with parameters: {'scaling_kind': 'std', 'state_rank': 2, 'lambda1': 0.13771899245388655, 'lambda2': 1.1054781155004125, 'coupling': 1.559951616237607, 'alpha1': 0.005793266037223135, 'alpha2': 0.005998392365776778, 'dyadic_order': 0, 'C': 1612.4591095265375} because of the following error: ValueError('Batchwise evaluation requires matching batch shapes; got (23, 1) and (1, 180).').
Traceback (most recent call last):
  File "/Home/stat/pelizzari/.local/lib/python3.12/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/Home/stat/pelizzari/Nonparametric/numerical_experiments/notebooks/VSIG_Github/run_classifiers.py", line 777, in objective
    G_train, resolved = compute_vsig_train_gram(
                        ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Home/stat/pelizzari/Nonparametric/numerical_experiments/notebooks/VSIG_Github/run_classifiers.py", line 628, in c

ValueError: Batchwise evaluation requires matching batch shapes; got (23, 1) and (1, 180).

In [6]:
jax.local_device_count()

8

In order to reproduce the experiments presented in the paper, run the following. (it takes a while as some of the datasets are big)

In [ ]:
!python3 run_classifiers.py